`[Cell 1]`

# Question B: from the raw Kaggle data to the minute and hourly datasets

**Question B:** Can we accurately forecast a zone's AC energy demand by combining historical indoor environmental conditions with external weather forecasts?

This notebook runs the whole pipeline in one go. Put it in the same folder as the 14 raw CSV files from Kaggle (`2018Floor1.csv` … `2019Floor7.csv`) and run all cells from top to bottom.

| Part | Steps |
|---|---|
| **1. General cleaning** (sections 1–5) | Load the 14 files, remove the invalid row, fix 9 impossible readings, flag them (`replaced?`), flag public holidays (`holiday?`), add outdoor weather (rainfall, temperature) |
| **2. Question B minute dataset** (sections 6–10) | Drop Light, Plug and Lux; drop Floor 1; add anomaly flags; keep the slide's time window; add outdoor humidity and solar radiation |
| **3. Question B hourly dataset** (sections 11–12) | One row per floor per hour, 54-minute rule |

**Outputs:** `question_b_hourly_data.parquet` (committed), plus `question_b_minute_data.parquet` and `question_b_hourly_data.csv` (made locally, not committed: see `.gitignore`).

**Internet:** the outdoor weather comes from the Open-Meteo historical archive. The first run downloads it and saves it as `weather_bangkok_daily.csv`, `weather_bangkok_hourly.csv`, `weather_bangkok_daily_humidity.csv`, `weather_bangkok_hourly_humidity.csv`, `weather_bangkok_daily_solar.csv` and `weather_bangkok_hourly_solar.csv`; later runs use the saved files and need no internet.

The cleaning decisions are summarised in `README.md` in this folder. Running all cells takes several minutes.

In [2]:
# [Cell 2]
import json
import os
import urllib.request

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
yes_no = {True: 'Yes', False: 'No'}
MIN_MINUTES = 54          # an hour's average is kept only if at least 54 of its 60 minutes have a reading

`[Cell 3]`

## 1. Load the 14 raw files

The raw data from Kaggle: one CSV per floor per year (`2018Floor1.csv` … `2019Floor7.csv`), one row per minute. Put them in the same folder as this notebook. A `Year` and `Floor` column is added to each file so every row can be traced back to its source.

In [3]:
# [Cell 4]
raw = {}
for year in [2018, 2019]:
    for floor in range(1, 8):
        df = pd.read_csv(f'{year}Floor{floor}.csv')
        df.insert(0, 'Year', year)
        df.insert(1, 'Floor', floor)
        raw[(year, floor)] = df

print({f'{y} F{f}': len(df) for (y, f), df in raw.items()})

{'2018 F1': 264960, '2018 F2': 264960, '2018 F3': 264960, '2018 F4': 264960, '2018 F5': 264960, '2018 F6': 264960, '2018 F7': 264960, '2019 F1': 525600, '2019 F2': 525600, '2019 F3': 525600, '2019 F4': 525600, '2019 F5': 525600, '2019 F6': 424169, '2019 F7': 525600}


`[Cell 5]`

## 2. Remove the invalid row and combine the files

The Floor 6, 2019 file ends on 2019-10-22 13:27 with one invalid row (`Date` is "2" and every reading is empty). It is removed. The files are then combined into one table and `Date` is converted to a date-time.

In [4]:
# [Cell 6]
f6 = raw[(2019, 6)]
invalid = pd.to_datetime(f6['Date'], errors='coerce').isna()
print('invalid rows removed from 2019 Floor 6:', invalid.sum())
raw[(2019, 6)] = f6[~invalid]

combined_2018 = pd.concat([raw[(2018, f)] for f in range(1, 8)], ignore_index=True)
combined_2019 = pd.concat([raw[(2019, f)] for f in range(1, 8)], ignore_index=True)
all_data = pd.concat([combined_2018, combined_2019], ignore_index=True)
all_data['Date'] = pd.to_datetime(all_data['Date'])
del raw, combined_2018, combined_2019

print(all_data.shape)
print('duplicate Year-Floor-Date rows (expected 0):', all_data.duplicated(subset=['Year', 'Floor', 'Date']).sum())
all_data.groupby(['Year', 'Floor'])['Date'].agg(['min', 'max', 'count'])

invalid rows removed from 2019 Floor 6: 1
(5432488, 51)
duplicate Year-Floor-Date rows (expected 0): 0


min                 max   count
Year Floor                                       
2018 1     2018-07-01 2018-12-31 23:59:00  264960
     2     2018-07-01 2018-12-31 23:59:00  264960
     3     2018-07-01 2018-12-31 23:59:00  264960
     4     2018-07-01 2018-12-31 23:59:00  264960
     5     2018-07-01 2018-12-31 23:59:00  264960
     6     2018-07-01 2018-12-31 23:59:00  264960
     7     2018-07-01 2018-12-31 23:59:00  264960
2019 1     2019-01-01 2019-12-31 23:59:00  525600
     2     2019-01-01 2019-12-31 23:59:00  525600
     3     2019-01-01 2019-12-31 23:59:00  525600
     4     2019-01-01 2019-12-31 23:59:00  525600
     5     2019-01-01 2019-12-31 23:59:00  525600
     6     2019-01-01 2019-10-22 13:27:00  424168
     7     2019-01-01 2019-12-31 23:59:00  525600

`[Cell 7]`

## 3. Fix the invalid readings

Nine impossible readings in six rows, found by checking each meter's highest readings against its normal range. Most were junk readings as a meter came back online after a gap, dropping straight back to normal the next minute:

| Column | Floor / Year | Minute | Value | Fix |
|---|---|---|---|---|
| `z3_Light(kW)` | 1 / 2018 | 2018-09-11 09:54 | 4762.43 | Average of the minutes before and after |
| `z4_Light(kW)` | 1 / 2018 | 2018-09-07 15:50 | 512.01 | Average of the minutes before and after |
| `z2_AC2(kW)` | 1 / 2018 | 2018-09-10 10:46 and 10:47 | 798.70, 798.72 | The first valid reading afterwards (0.02) |
| `z2_AC3(kW)` | 1 / 2018 | 2018-09-07 15:50 | 24.95 | The next minute's reading |
| `z2_AC13(kW)` | 2 / 2018 | 2018-08-03 12:04 | 10.30 | The next minute's reading |
| `z1_AC2(kW)`, `z1_AC3(kW)`, `z5_Light(kW)` | 4 / 2018 | 2018-11-21 15:20 | 20.28, 9.91, 58.61 | The next minute's reading |

Each fixed row is marked in a new column, `replaced?`.

In [5]:
# [Cell 8]
def row_of(year, floor, minute):
    """Index of the row for this year, floor and minute."""
    match = all_data.index[(all_data['Year'] == year) & (all_data['Floor'] == floor)
                           & (all_data['Date'] == pd.Timestamp(minute))]
    return match[0]

# lighting glitches: average of the minutes before and after
for column, floor, minute in [('z3_Light(kW)', 1, '2018-09-11 09:54'), ('z4_Light(kW)', 1, '2018-09-07 15:50')]:
    i = row_of(2018, floor, minute)
    all_data.loc[i, column] = (all_data.loc[i - 1, column] + all_data.loc[i + 1, column]) / 2

# start-up readings after an outage: the first valid reading afterwards
for minute in ['2018-09-10 10:46', '2018-09-10 10:47']:
    all_data.loc[row_of(2018, 1, minute), 'z2_AC2(kW)'] = 0.02

# junk first readings after a gap: the next minute's reading
for columns, floor, minute in [(['z2_AC3(kW)'], 1, '2018-09-07 15:50'),
                               (['z2_AC13(kW)'], 2, '2018-08-03 12:04'),
                               (['z1_AC2(kW)', 'z1_AC3(kW)', 'z5_Light(kW)'], 4, '2018-11-21 15:20')]:
    i = row_of(2018, floor, minute)
    for column in columns:
        all_data.loc[i, column] = all_data.loc[i + 1, column]

replaced_rows = pd.DataFrame({
    'Year':  [2018] * 6,
    'Floor': [1, 1, 1, 1, 2, 4],
    'Date':  pd.to_datetime(['2018-09-07 15:50', '2018-09-10 10:46', '2018-09-10 10:47',
                             '2018-09-11 09:54', '2018-08-03 12:04', '2018-11-21 15:20']),
})
is_replaced = pd.MultiIndex.from_frame(all_data[['Year', 'Floor', 'Date']]).isin(pd.MultiIndex.from_frame(replaced_rows))
all_data['replaced?'] = pd.Series(is_replaced, index=all_data.index).map(yes_no)

print('rows marked replaced?:', (all_data['replaced?'] == 'Yes').sum())
all_data[['z3_Light(kW)', 'z4_Light(kW)', 'z2_AC2(kW)', 'z2_AC3(kW)', 'z2_AC13(kW)']].max()

rows marked replaced?: 6


z3_Light(kW)    232.71
z4_Light(kW)    186.08
z2_AC2(kW)       80.92
z2_AC3(kW)        2.08
z2_AC13(kW)       2.16
dtype: float64

`[Cell 9]`

## 4. Flag public holidays

`holiday?` marks the weekdays when the office was closed (or largely closed) for a holiday. The approach:

1. **List every holiday** (next cell): every **weekday** between July 2018 and December 2019 that the sources list as a holiday in Thailand, including substitution days (when a holiday falls on a weekend, the next weekday is a day off instead) and the Cabinet's extra days. Holidays on a Saturday or Sunday are not listed: those days are weekends anyway.
2. **Look at the AC load** on each of them, compared with a normal weekday.
3. **Keep it as a holiday unless the AC load was as high as on a normal working day** (that would mean the office was open as usual).

**Sources:**
- [timeanddate.com: Holidays and Observances in Thailand in 2018](https://www.timeanddate.com/holidays/thailand/2018)
- [timeanddate.com: Holidays and Observances in Thailand in 2019](https://www.timeanddate.com/holidays/thailand/2019) (lists the substitution days)
- [Bangkok Post: Cabinet approves Dec 30 as additional New Year holiday (2019)](https://www.bangkokpost.com/thailand/general/1782464/cabinet-approves-dec-30-as-additional-ny-holiday) (the Cabinet's extra day, 30 December 2019)

In [6]:
# [Cell 10]
# 1. Every weekday listed as a holiday in Thailand, July 2018 - December 2019
holidays = pd.DataFrame([
    # 2018
    ('2018-07-27', 'Asalha Bucha'),
    ('2018-07-30', "Day off for King Vajiralongkorn's birthday"),
    ('2018-08-13', "Day off for the Queen Mother's birthday"),
    ('2018-10-15', 'Day off for the anniversary of the death of King Bhumibol'),
    ('2018-10-23', 'Chulalongkorn Day'),
    ('2018-12-05', "King Bhumibol's birthday"),
    ('2018-12-10', 'Constitution Day'),
    ('2018-12-25', 'Christmas Day'),
    ('2018-12-31', "New Year's Eve"),
    # 2019
    ('2019-01-01', "New Year's Day"),
    ('2019-02-19', 'Makha Bucha'),
    ('2019-04-08', 'Day off for Chakri Day'),
    ('2019-04-12', 'Songkran (observed)'),
    ('2019-04-15', 'Songkran'),
    ('2019-04-16', 'Songkran (observed)'),
    ('2019-05-01', 'Labour Day'),
    ('2019-05-06', 'Coronation Day (day off)'),
    ('2019-05-09', 'Royal Ploughing Ceremony Day'),
    ('2019-05-20', 'Day off for Visakha Bucha'),
    ('2019-06-03', "Queen Suthida's birthday"),
    ('2019-07-16', 'Asalha Bucha'),
    ('2019-07-17', 'Buddhist Lent Day'),
    ('2019-07-29', "Day off for King Vajiralongkorn's birthday"),
    ('2019-08-12', "Queen Mother's birthday"),
    ('2019-10-14', 'Day off for the anniversary of the death of King Bhumibol'),
    ('2019-10-23', 'Chulalongkorn Day'),
    ('2019-12-05', "King Bhumibol's birthday"),
    ('2019-12-10', 'Constitution Day'),
    ('2019-12-30', 'New Year special holiday (added by the Cabinet)'),
    ('2019-12-31', "New Year's Eve"),
], columns=['Day', 'Holiday'])
holidays['Day'] = pd.to_datetime(holidays['Day'])
holidays = holidays.set_index('Day')

print('holidays listed:', len(holidays), '| any on a weekend (expected False):', (holidays.index.dayofweek >= 5).any())

holidays listed: 30 | any on a weekend (expected False): False


`[Cell 11]`

**2. Look at the AC load on each holiday**

For every weekday, the next cell works out how much AC the office floors (2–7) used compared with a normal weekday:

1. Add up each floor's AC units and take the average between 08:00 and 17:00 that day.
2. Divide by that floor's **typical weekday** (its median weekday). 100% means the usual amount of AC; 5% means one-twentieth of it.
3. Take the median across Floors 2–7, so one unusual floor does not decide the day.

Floor 1 is left out: its AC runs around the clock and does not follow the office schedule.

**3. The rule:** a listed holiday is kept as a holiday **unless its AC load was at least 75% of a normal weekday**, the level of an ordinary working day (most weekdays are at 75% or more). At that level the office was clearly open as usual.

In [7]:
# [Cell 12]
NORMAL_DAY_LEVEL = 0.75        # at or above this share of a normal weekday, the office was open as usual

ac_cols = [c for c in all_data.columns if '_AC' in c]
office = all_data[(all_data['Floor'] >= 2) & (all_data['Date'].dt.hour >= 8) & (all_data['Date'].dt.hour < 17)]

# each floor's average AC load (all units added up) on each day, 08:00-17:00
floor_day = (office[ac_cols].sum(axis=1, min_count=1)
             .groupby([office['Floor'], office['Date'].dt.normalize().rename('Day')]).mean()
             .rename('AC total').reset_index().dropna())

# each floor's typical weekday, then each day as a share of it; the median across floors for each day
normal = floor_day[floor_day['Day'].dt.dayofweek < 5].groupby('Floor')['AC total'].median()
floor_day['Share of normal'] = floor_day['AC total'] / floor_day['Floor'].map(normal)
share_of_normal = floor_day.groupby('Day')['Share of normal'].median()

# each listed holiday: its AC load and the decision
holiday_check = holidays.copy()
holiday_check['AC load (% of a normal weekday)'] = (share_of_normal.reindex(holiday_check.index) * 100).round(1)
holiday_check['Holiday?'] = np.where(share_of_normal.reindex(holiday_check.index) < NORMAL_DAY_LEVEL,
                                     'Yes', 'No (office open as usual)')

print(holiday_check['Holiday?'].value_counts().to_string())
holiday_check

Holiday?
Yes                          28
No (office open as usual)     2


,Holiday,AC load (% of a normal weekday),Holiday?
Day,,,
2018-07-27,Asalha Bucha,4.8,Yes
2018-07-30,Day off for King Vajiralongkorn's birthday,0.0,Yes
2018-08-13,Day off for the Queen Mother's birthday,0.0,Yes
2018-10-15,Day off for the anniversary of the death of Ki...,0.0,Yes
2018-10-23,Chulalongkorn Day,0.0,Yes
2018-12-05,King Bhumibol's birthday,0.0,Yes
2018-12-10,Constitution Day,0.2,Yes
2018-12-25,Christmas Day,99.8,No (office open as usual)
2018-12-31,New Year's Eve,0.0,Yes


`[Cell 13]`

**What the check shows:**

- On most holidays the AC was essentially off (5% of a normal weekday or less): the office was closed.
- **Royal Ploughing Ceremony Day (9 May 2019)** was at about 34%: the office was partly in use, but well below a normal working day, so it is kept as a holiday.
- On **Christmas Day 2018** (about 100%) and **Labour Day 2019** (about 131%) the AC ran as on a normal working day: the office was open, so they are not holidays here.

Any cut-off between 35% and 99% of a normal weekday gives exactly the same result.

In [8]:
# [Cell 14]
confirmed_holidays = holiday_check.index[holiday_check['Holiday?'] == 'Yes']
all_data['holiday?'] = all_data['Date'].dt.normalize().isin(confirmed_holidays).map(yes_no)

print('holidays flagged:', len(confirmed_holidays))
all_data['holiday?'].value_counts()           # rows = minutes on each floor

holidays flagged: 28


holiday?
No     5157448
Yes     275040
Name: count, dtype: int64

`[Cell 15]`

## 5. Add outdoor weather (rainfall and temperature)

For Bangkok (Chulalongkorn University, where the building is), from the Open-Meteo historical archive. This is observed historical weather (from a weather model for the location, not a station at the building), used as a stand-in for weather forecasts.

- `rainfall (mm)`, `outdoor temp (degC)`: the day's total rainfall and average temperature. Every minute of a day gets that day's value.
- `hourly rainfall (mm)`: the rain that fell during that clock hour. Open-Meteo stamps rain at the end of the hour, so the rain for 14:00–14:59 is the value stamped 15:00.
- `hourly outdoor temp (degC)`: the temperature at the start of the hour. Every minute of an hour gets that hour's value.

In [9]:
# [Cell 16]
base_url = ('https://archive-api.open-meteo.com/v1/archive?latitude=13.7367&longitude=100.5332'
            '&timezone=Asia%2FBangkok')

daily_file = 'weather_bangkok_daily.csv'
if not os.path.exists(daily_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2019-12-31'
                                         '&daily=precipitation_sum,temperature_2m_mean', timeout=60))['daily']
    pd.DataFrame({'Day': w['time'], 'rainfall (mm)': w['precipitation_sum'],
                  'outdoor temp (degC)': w['temperature_2m_mean']}).to_csv(daily_file, index=False)

hourly_file = 'weather_bangkok_hourly.csv'
if not os.path.exists(hourly_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2020-01-01'
                                         '&hourly=precipitation,temperature_2m', timeout=60))['hourly']
    pd.DataFrame({'Time': w['time'], 'precipitation (mm)': w['precipitation'],
                  'temperature (degC)': w['temperature_2m']}).to_csv(hourly_file, index=False)

daily_weather = pd.read_csv(daily_file, parse_dates=['Day']).set_index('Day')
hourly_weather = pd.read_csv(hourly_file, parse_dates=['Time']).set_index('Time')

day = all_data['Date'].dt.normalize()
hour = all_data['Date'].dt.floor('h')
all_data['rainfall (mm)'] = day.map(daily_weather['rainfall (mm)'])
all_data['outdoor temp (degC)'] = day.map(daily_weather['outdoor temp (degC)'])
all_data['hourly rainfall (mm)'] = hour.map(hourly_weather['precipitation (mm)'].shift(-1))
all_data['hourly outdoor temp (degC)'] = hour.map(hourly_weather['temperature (degC)'])

print('cleaned data:', all_data.shape)
all_data[['rainfall (mm)', 'outdoor temp (degC)', 'hourly rainfall (mm)', 'hourly outdoor temp (degC)']].describe().round(1)

cleaned data: (5432488, 57)


,rainfall (mm),outdoor temp (degC),hourly rainfall (mm),hourly outdoor temp (degC)
count,5432488.0,5432488.0,5432488.0,5432488.0
mean,3.8,28.2,0.2,28.2
std,5.8,1.5,0.7,2.8
min,0.0,21.8,0.0,16.7
25%,0.1,27.3,0.0,26.2
50%,1.2,28.1,0.0,28.0
75%,5.1,29.0,0.0,30.1
max,42.1,32.9,14.7,38.8


`[Cell 17]`

## 6. Question B: drop the unused columns and Floor 1

- Light, Plug (electricity) and Lux are not used for Question B.
- Floor 1 has no indoor temperature or humidity sensors.

In [ ]:
# [Cell 18]
unused = [c for c in all_data.columns if 'Light' in c or 'Plug' in c or '(lux)' in c]
data_b = all_data.drop(columns=unused)
data_b = data_b[data_b['Floor'] != 1].reset_index(drop=True)
# floor 1 ac removed 
print('columns dropped:', unused)
print(data_b.shape)

columns dropped: ['z1_Light(kW)', 'z1_Plug(kW)', 'z2_Light(kW)', 'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_Light(kW)', 'z1_S1(lux)', 'z2_S1(lux)', 'z3_S1(lux)', 'z4_Plug(kW)', 'z4_S1(lux)', 'z5_Light(kW)', 'z5_Plug(kW)', 'z5_S1(lux)']
(4641928, 42)


`[Cell 19]`

## 7. Anomaly flags (values kept)

- `AC spike?`: any AC unit on that floor read more than 1.5 × that unit's 99.9th percentile (worked out per floor and unit over the full record, before the time window is applied). `AC spike units` lists which units. For units that are almost always off (99.9th percentile = 0 kW), any reading above 0 counts, so many flags are real (rare) use, not errors.
- `indoor temp out of range?` / `indoor humidity out of range?`: any zone's S1 temperature outside 15–36 °C, or humidity outside 20–100%.
- `z1_S1 supply-air sensor?`: 'Yes' on every Floor 5 row; Floor 5's zone 1 temperature sensor reads AC supply air, not room air.

The readings themselves are not changed.

In [11]:
# [Cell 20]
ac_cols = [c for c in data_b.columns if '_AC' in c]
spike_units = pd.Series('', index=data_b.index)
for floor, g in data_b.groupby('Floor'):
    for col in ac_cols:
        if g[col].isna().all():
            continue                                  # this floor does not have this unit
        limit = 1.5 * g[col].quantile(0.999)
        spike_units.loc[g.index[g[col] > limit]] += col + ' '
data_b['AC spike?'] = (spike_units != '').map(yes_no)
data_b['AC spike units'] = spike_units.str.strip()

temp_cols = [c for c in data_b.columns if c.endswith('S1(degC)')]
rh_cols = [c for c in data_b.columns if c.endswith('S1(RH%)')]
data_b['indoor temp out of range?'] = ((data_b[temp_cols] < 15) | (data_b[temp_cols] > 36)).any(axis=1).map(yes_no)
data_b['indoor humidity out of range?'] = ((data_b[rh_cols] < 20) | (data_b[rh_cols] > 100)).any(axis=1).map(yes_no)
data_b['z1_S1 supply-air sensor?'] = (data_b['Floor'] == 5).map(yes_no)

for flag in ['AC spike?', 'indoor temp out of range?', 'indoor humidity out of range?']:
    print(f'rows with {flag}', (data_b[flag] == 'Yes').sum())

rows with AC spike? 1536
rows with indoor temp out of range? 33972
rows with indoor humidity out of range? 0


`[Cell 21]`

## 8. Keep the slide's time window

Main modelling: March–December 2019. Extra training data: July–September 2018. The rest has no indoor temperature or humidity readings.

In [12]:
# [Cell 22]
in_window = (data_b['Date'].between('2018-07-01 00:00', '2018-09-30 23:59')
             | data_b['Date'].between('2019-03-01 00:00', '2019-12-31 23:59'))
data_b = data_b[in_window].reset_index(drop=True)
data_b.groupby(['Year', 'Floor'])['Date'].agg(['min', 'max', 'count'])

min                 max   count
Year Floor                                       
2018 2     2018-07-01 2018-09-30 23:59:00  132480
     3     2018-07-01 2018-09-30 23:59:00  132480
     4     2018-07-01 2018-09-30 23:59:00  132480
     5     2018-07-01 2018-09-30 23:59:00  132480
     6     2018-07-01 2018-09-30 23:59:00  132480
     7     2018-07-01 2018-09-30 23:59:00  132480
2019 2     2019-03-01 2019-12-31 23:59:00  440640
     3     2019-03-01 2019-12-31 23:59:00  440640
     4     2019-03-01 2019-12-31 23:59:00  440640
     5     2019-03-01 2019-12-31 23:59:00  440640
     6     2019-03-01 2019-10-22 13:27:00  339208
     7     2019-03-01 2019-12-31 23:59:00  440640

`[Cell 23]`

## 9. Add outdoor humidity and solar radiation

From the same Open-Meteo archive, in the same two forms as temperature and rainfall:

- `outdoor humidity (%)`: the day's average; `hourly outdoor humidity (%)`: the reading at the start of the hour.
- `solar radiation (MJ/m2)`: the day's total sunshine energy; `hourly solar radiation (W/m2)`: the average during that clock hour. Open-Meteo stamps solar radiation at the end of the hour (like rainfall), so the value for 14:00–14:59 is the one stamped 15:00.

Daily values go on every minute of the day, hourly values on every minute of the hour.

In [13]:
# [Cell 24]
daily_humidity_file = 'weather_bangkok_daily_humidity.csv'
if not os.path.exists(daily_humidity_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2020-01-01'
                                         '&daily=relative_humidity_2m_mean', timeout=60))['daily']
    pd.DataFrame({'Day': w['time'], 'relative humidity (%)': w['relative_humidity_2m_mean']}).to_csv(daily_humidity_file, index=False)

hourly_humidity_file = 'weather_bangkok_hourly_humidity.csv'
if not os.path.exists(hourly_humidity_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2020-01-01'
                                         '&hourly=relative_humidity_2m', timeout=60))['hourly']
    pd.DataFrame({'Time': w['time'], 'relative humidity (%)': w['relative_humidity_2m']}).to_csv(hourly_humidity_file, index=False)

daily_humidity = pd.read_csv(daily_humidity_file, parse_dates=['Day']).set_index('Day')['relative humidity (%)']
hourly_humidity = pd.read_csv(hourly_humidity_file, parse_dates=['Time']).set_index('Time')['relative humidity (%)']

data_b.insert(data_b.columns.get_loc('outdoor temp (degC)') + 1, 'outdoor humidity (%)',
              data_b['Date'].dt.normalize().map(daily_humidity))
data_b.insert(data_b.columns.get_loc('hourly outdoor temp (degC)') + 1, 'hourly outdoor humidity (%)',
              data_b['Date'].dt.floor('h').map(hourly_humidity))

daily_solar_file = 'weather_bangkok_daily_solar.csv'
if not os.path.exists(daily_solar_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2020-01-01'
                                         '&daily=shortwave_radiation_sum', timeout=60))['daily']
    pd.DataFrame({'Day': w['time'], 'solar radiation (MJ/m2)': w['shortwave_radiation_sum']}).to_csv(daily_solar_file, index=False)

hourly_solar_file = 'weather_bangkok_hourly_solar.csv'
if not os.path.exists(hourly_solar_file):
    w = json.load(urllib.request.urlopen(base_url + '&start_date=2018-07-01&end_date=2020-01-01'
                                         '&hourly=shortwave_radiation', timeout=60))['hourly']
    pd.DataFrame({'Time': w['time'], 'solar radiation (W/m2)': w['shortwave_radiation']}).to_csv(hourly_solar_file, index=False)

daily_solar = pd.read_csv(daily_solar_file, parse_dates=['Day']).set_index('Day')['solar radiation (MJ/m2)']
hourly_solar = pd.read_csv(hourly_solar_file, parse_dates=['Time']).set_index('Time')['solar radiation (W/m2)']
solar_in_hour = hourly_solar.shift(-1)                 # stamped at the end of the hour

data_b.insert(data_b.columns.get_loc('outdoor humidity (%)') + 1, 'solar radiation (MJ/m2)',
              data_b['Date'].dt.normalize().map(daily_solar))
data_b.insert(data_b.columns.get_loc('hourly outdoor humidity (%)') + 1, 'hourly solar radiation (W/m2)',
              data_b['Date'].dt.floor('h').map(solar_in_hour))

weather_cols = ['rainfall (mm)', 'outdoor temp (degC)', 'outdoor humidity (%)', 'solar radiation (MJ/m2)',
                'hourly rainfall (mm)', 'hourly outdoor temp (degC)', 'hourly outdoor humidity (%)',
                'hourly solar radiation (W/m2)']
print('missing weather values:', data_b[weather_cols].isna().sum().sum())
data_b[weather_cols].describe().T.round(1)

missing weather values: 0


,count,mean,std,min,25%,50%,75%,max
rainfall (mm),3337288.0,4.5,6.2,0.0,0.2,2.1,6.4,42.1
outdoor temp (degC),3337288.0,28.4,1.6,21.8,27.5,28.3,29.2,32.9
outdoor humidity (%),3337288.0,75.9,8.8,49.0,71.0,78.0,82.0,90.0
solar radiation (MJ/m2),3337288.0,17.5,4.2,5.8,14.9,18.4,20.7,25.2
hourly rainfall (mm),3337288.0,0.2,0.7,0.0,0.0,0.0,0.1,14.7
hourly outdoor temp (degC),3337288.0,28.5,2.7,16.7,26.5,28.2,30.2,38.8
hourly outdoor humidity (%),3337288.0,75.9,14.1,23.0,68.0,79.0,87.0,100.0
hourly solar radiation (W/m2),3337288.0,202.5,272.6,0.0,0.0,16.0,384.0,964.0


`[Cell 25]`

## 10. Save the minute-level dataset

`question_b_minute_data.parquet`: one row per floor per minute. Not saved as CSV: with over 3 million rows it is larger than Excel can open.

In [14]:
# [Cell 26]
data_b.to_parquet('question_b_minute_data.parquet', index=False)
print(data_b.shape, '| saved and reloaded OK:', pd.read_parquet('question_b_minute_data.parquet').equals(data_b))

(3337288, 51) | saved and reloaded OK: True


`[Cell 27]`

## 11. Group by hour

One row per floor per hour. `Hour` is the start of the hour (full date and time).

- **Meters and sensors:** the average of the hour's minutes, **only if at least 54 of the 60 minutes have a reading**; otherwise `NaN`. No gaps are filled.
- **Values that are already one per hour or per day** (weather, `holiday?`, `z1_S1 supply-air sensor?`): kept as they are.
- **Flags:** 'Yes' if any minute in the hour was 'Yes'.
- `minutes in hour`: how many minutes of the hour exist in the data (60, except where a file ends mid-hour).

In [15]:
# [Cell 28]
data = data_b.copy()
data['Hour'] = data['Date'].dt.floor('h')

readings = [c for c in data.columns if '_AC' in c or '_S1(' in c]       # AC meters and indoor sensors
kept_values = ['Year', 'holiday?', 'rainfall (mm)', 'outdoor temp (degC)', 'outdoor humidity (%)', 'solar radiation (MJ/m2)',
               'hourly rainfall (mm)', 'hourly outdoor temp (degC)', 'hourly outdoor humidity (%)',
               'hourly solar radiation (W/m2)',
               'z1_S1 supply-air sensor?']
any_minute_flags = ['replaced?', 'AC spike?', 'indoor temp out of range?', 'indoor humidity out of range?']

other = set(data.columns) - set(readings + kept_values + any_minute_flags + ['Floor', 'Date', 'Hour'] + ['AC spike units'])
print('readings:', len(readings), '| kept values:', len(kept_values), '| flags:', len(any_minute_flags))
print('columns not handled (expected: none):', other)

readings: 33 | kept values: 11 | flags: 4
columns not handled (expected: none): set()


In [16]:
# [Cell 29]
groups = data.groupby(['Floor', 'Hour'])

averages = groups[readings].mean()
minutes_recorded = groups[readings].count()
hourly_readings = averages.where(minutes_recorded >= MIN_MINUTES)        # NaN if fewer than 54 minutes

hourly_kept = groups[kept_values].first()
hourly_flags = pd.DataFrame({
    flag: (data[flag] == 'Yes').groupby([data['Floor'], data['Hour']]).any().map(yes_no)
    for flag in any_minute_flags
})

# the units that spiked during the hour
spiking = data.loc[data['AC spike units'] != '', ['Floor', 'Hour', 'AC spike units']]
spike_units = (spiking.groupby(['Floor', 'Hour'])['AC spike units']
               .agg(lambda s: ' '.join(sorted(set(' '.join(s).split())))))
hourly_flags['AC spike units'] = spike_units.reindex(hourly_flags.index, fill_value='')

minutes_in_hour = groups.size().rename('minutes in hour')

hourly = pd.concat([hourly_kept, minutes_in_hour, hourly_readings, hourly_flags], axis=1).reset_index()

# same column order as the minute-level data, with Hour in place of Date
order = [c if c != 'Date' else 'Hour' for c in data.columns if c != 'Hour']
order.insert(order.index('Hour') + 1, 'minutes in hour')
hourly = hourly[order]

set_to_nan = ((minutes_recorded > 0) & (minutes_recorded < MIN_MINUTES)).sum().sum()
print('hourly values set to NaN by the 54-minute rule:', set_to_nan)
print(hourly.shape)

hourly values set to NaN by the 54-minute rule: 19109
(55622, 52)


`[Cell 30]`

## 12. Checks on the hourly dataset

In [17]:
# [Cell 31]
# Rows per floor and year (one row per hour; Floor 6's 2019 file ends on 22 Oct 2019)
hourly.groupby(['Year', 'Floor'])['Hour'].agg(['min', 'max', 'count'])

min                 max  count
Year Floor                                      
2018 2     2018-07-01 2018-09-30 23:00:00   2208
     3     2018-07-01 2018-09-30 23:00:00   2208
     4     2018-07-01 2018-09-30 23:00:00   2208
     5     2018-07-01 2018-09-30 23:00:00   2208
     6     2018-07-01 2018-09-30 23:00:00   2208
     7     2018-07-01 2018-09-30 23:00:00   2208
2019 2     2019-03-01 2019-12-31 23:00:00   7344
     3     2019-03-01 2019-12-31 23:00:00   7344
     4     2019-03-01 2019-12-31 23:00:00   7344
     5     2019-03-01 2019-12-31 23:00:00   7344
     6     2019-03-01 2019-10-22 13:00:00   5654
     7     2019-03-01 2019-12-31 23:00:00   7344

In [18]:
# [Cell 32]
# Hours that do not have all 60 minutes in the data (expected: only Floor 6's last hour, 22 Oct 2019 13:00)
hourly[hourly['minutes in hour'] < 60][['Floor', 'Hour', 'minutes in hour']]

,Floor,Hour,minutes in hour
46069,6,2019-10-22 13:00:00,28


In [19]:
# [Cell 33]
# Spot check: recompute one hour by hand from the minute data and compare
floor, hour, column = 2, pd.Timestamp('2019-04-03 14:00'), 'z2_AC1(kW)'
by_hand = data.loc[(data['Floor'] == floor) & (data['Hour'] == hour), column].mean()
in_table = hourly.loc[(hourly['Floor'] == floor) & (hourly['Hour'] == hour), column].iloc[0]
print(f'{column}, Floor {floor}, {hour}: by hand {by_hand:.4f} | hourly table {in_table:.4f}')

z2_AC1(kW), Floor 2, 2019-04-03 14:00:00: by hand 30.4433 | hourly table 30.4433


In [20]:
# [Cell 34]
# How many hours carry each flag
flags = ['AC spike?', 'indoor temp out of range?', 'indoor humidity out of range?',
         'z1_S1 supply-air sensor?', 'replaced?', 'holiday?']
pd.DataFrame({f: hourly[f].value_counts() for f in flags}).fillna(0).astype(int)

,AC spike?,indoor temp out of range?,indoor humidity out of range?,z1_S1 supply-air sensor?,replaced?,holiday?
No,55427,54462,55622,46070,55621,52718
Yes,195,1160,0,9552,1,2904


`[Cell 35]`

## 13. Save the hourly dataset

`question_b_hourly_data.parquet` (for Python) and `question_b_hourly_data.csv` (to open in Excel).

In [21]:
# [Cell 36]
hourly.to_parquet('question_b_hourly_data.parquet', index=False)
hourly.to_csv('question_b_hourly_data.csv', index=False)

check = pd.read_parquet('question_b_hourly_data.parquet')
print(check.shape, '| saved and reloaded OK:', check.equals(hourly))

(55622, 52) | saved and reloaded OK: True


In [22]:
hourly.head()

,Year,Floor,Hour,minutes in hour,z2_AC1(kW),z2_AC2(kW),z2_AC3(kW),z2_AC4(kW),z1_AC1(kW),z1_S1(degC),z1_S1(RH%),z2_AC5(kW),z2_AC6(kW),z2_AC7(kW),z2_AC8(kW),z2_AC9(kW),z2_AC10(kW),z2_AC11(kW),z2_AC12(kW),z2_AC13(kW),z2_AC14(kW),z2_S1(degC),z2_S1(RH%),z3_S1(degC),z3_S1(RH%),z4_AC1(kW),z4_S1(degC),z4_S1(RH%),z1_AC2(kW),z1_AC3(kW),z1_AC4(kW),z5_AC1(kW),z5_S1(degC),z5_S1(RH%),z4_AC2(kW),z4_AC3(kW),z4_AC4(kW),replaced?,holiday?,rainfall (mm),outdoor temp (degC),outdoor humidity (%),solar radiation (MJ/m2),hourly rainfall (mm),hourly outdoor temp (degC),hourly outdoor humidity (%),hourly solar radiation (W/m2),AC spike?,AC spike units,indoor temp out of range?,indoor humidity out of range?,z1_S1 supply-air sensor?
0,2018,2,2018-07-01 00:00:00,60,1.031833,0.0,0.0,0.0,0.0,NaN,NaN,1.269167,1.204667,0.853667,0.877500,0.946000,1.028333,0.0,0.0,0.0,0.0,28.824167,64.771000,29.263103,67.830690,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,0.1,28.8,74,22.76,0.0,27.1,88,0.0,No,,No,No,No
1,2018,2,2018-07-01 01:00:00,60,1.040500,0.0,0.0,0.0,0.0,28.402000,59.545500,1.280000,1.216333,0.859333,0.885167,0.953333,1.036500,0.0,0.0,0.0,0.0,28.873833,64.581167,29.386167,68.082667,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,0.1,28.8,74,22.76,0.0,26.8,88,0.0,No,,No,No,No
2,2018,2,2018-07-01 02:00:00,60,1.040833,0.0,0.0,0.0,0.0,28.416500,59.506833,1.279667,1.215000,0.859667,0.883333,0.956000,1.037833,0.0,0.0,0.0,0.0,28.893167,64.506000,29.384000,68.631833,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,0.1,28.8,74,22.76,0.0,26.5,86,0.0,No,,No,No,No
3,2018,2,2018-07-01 03:00:00,60,1.044833,0.0,0.0,0.0,0.0,28.437000,59.792333,1.275000,1.216833,0.860333,0.889500,0.955500,1.040000,0.0,0.0,0.0,0.0,28.897167,64.412000,29.315167,68.344667,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,0.1,28.8,74,22.76,0.0,25.9,84,0.0,No,,No,No,No
4,2018,2,2018-07-01 04:00:00,60,1.048167,0.0,0.0,0.0,0.0,28.437414,59.829483,1.277833,1.220500,0.864667,0.894500,0.957500,1.043833,0.0,0.0,0.0,0.0,28.912414,64.168448,29.387667,68.408000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,0.1,28.8,74,22.76,0.0,25.5,84,0.0,No,,No,No,No
